# P03 · AAM on the paper test set

Generated by `paper/make_notebooks.py` from the archived notebook in `experimented/`. Paper settings: larger test set (>= 100k unique test tokens per language), unseen languages ja / ru (eval only), per-item results, seeds; shared data and reference rows come from the P20a / P20b kernels. See `paper/README.md`.

In [ ]:
# Colab: Runtime -> Change runtime type -> T4 GPU.  Kaggle: Settings -> Accelerator GPU T4/P100, Internet ON.
!pip -q install -U transformers datasets pandas matplotlib "pyarrow>=19,<24"
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Settings
Keep the same settings (and `data_dir`) in every notebook, so data, reference rows and results are shared and the final table compares all algorithms.

In [ ]:
import os
CFG = dict(
    model="Qwen/Qwen3-0.6B-Base",      # any Qwen3 base model, e.g. "Qwen/Qwen3-1.7B-Base" (needs more GPU memory)
    # data + results folder, shared by all notebooks. Colab: Google Drive (mounted when data loads), so the original /
    # drop reference rows are computed ONCE and every algorithm lands in one table. Kaggle: /kaggle/working.
    data_dir=("/kaggle/working/layer_merging" if os.path.exists("/kaggle/working") else
              "/content/drive/MyDrive/layer_merging" if os.path.exists("/content") else "data"),
    langs=["en", "hi", "fr", "es", "de", "zh"],   # also available: "ja", "ru"
    merge_counts=[2, 4],            # layers removed (= merged pairs); 6 of 28 layers = 21 %
    protect=2,                         # never merge the first / last 2 layers
    # calibration / training text: Wiki-40B train
    seq=256, n_train=300, n_calib=16,  # sequences per language (the last calibration sequence is held out)
    # test sets, per language (few-shot everywhere)
    n_mc4=16, n_wiki=16,               # perplexity sequences
    n_mmmlu=100, n_include=100, n_mgsm=40, n_bele=100, n_xnli=100,
    shots=dict(mmmlu=5, include=5, mgsm=4, belebele=3, xnli=3),
    max_len=2048, mgsm_tokens=160,     # prompt length cap (oldest shots are cut), MGSM answer length
    ridge=1e-2,                        # ridge strength of all least-squares fits
    steps=400,                         # training steps (each: one sequence per language)
    lr=5e-5,
    anneal_frac=0.7,                   # fraction of steps over which ALPHA goes 1 -> 0; 0 = no annealing (ablation)
    hid_w=1.0,                         # hidden-state loss weight next to the KL loss
)
if os.environ.get("SMOKE"):  # tiny local test
    CFG.update(data_dir=os.environ["SMOKE"], seq=64, n_train=8, n_calib=3, n_mc4=2, n_wiki=2, n_mmmlu=4, n_include=4,
               n_mgsm=2, n_bele=4, n_xnli=3, mgsm_tokens=24, merge_counts=[1, 2], steps=4, n_search=2)

In [ ]:
# ---- Paper runs (generated by paper/make_notebooks.py) ----
# Larger test set: >= 100k unique test tokens per language (counted and printed by p20a), per-item results (for
# bootstrap CIs / paired tests), two languages never used for calibration or training (ja, ru), seeds.
CFG.update(
    merge_counts=[2, 4],
    n_mc4=128, n_wiki=128,                      # 128 x 256 = 32k tokens each
    n_mmmlu=200, n_include=200, n_mgsm=250, n_bele=200, n_xnli=200,   # MGSM: all 250 test problems
    unseen=["ja", "ru"], eval_unseen=False,     # True: also evaluate ja / ru (never calibrated or trained on); +25 % eval time
    seed=0, tag="",                             # tag: suffix of the result name (seeds, ablations)
    math_frac=0.0,                              # share of each language's training sequences replaced by GSM8K-train text
    lr_new=2e-4, lr_smear=1e-2, xtc_rank=64,    # SKM / XTC (new modules)
)
CFG.update({})
# Shared inputs from the kernels this one depends on (data cache + original / drop reference rows).
import glob, json, shutil
os.makedirs(CFG["data_dir"], exist_ok=True)
for _f in sorted(glob.glob("/kaggle/input/**/layer_merging/*", recursive=True)):
    _d = os.path.join(CFG["data_dir"], os.path.basename(_f))
    if _f.endswith(".pt") and not os.path.exists(_d):
        shutil.copy(_f, _d)
        print("shared data:", _f)
    elif os.path.basename(_f).startswith("results_reference_"):
        _R = json.load(open(_d)) if os.path.exists(_d) else {}
        _R.update(json.load(open(_f)))
        json.dump(_R, open(_d, "w"), indent=1)
        print("shared reference rows:", _f)

## Data
Calibration/training: Wiki-40B train. Tests (never used for fitting): **MMMLU** (English: MMLU) 5-shot, **INCLUDE** 5-shot, **MGSM** 4-shot chain of thought, **Belebele** 3-shot, **XNLI** 3-shot, plus mC4 and Wiki-40B test perplexity. Few-shot examples come from dev/validation/train splits, or from rows held out of evaluation.

In [ ]:
import ast, re, json, math, time, copy, gc, glob, random, hashlib, contextlib
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer

try:
    display
except NameError:
    display = print
torch.manual_seed(CFG["seed"])
DEV = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
LANGS, SLUG = CFG["langs"], CFG["model"].split("/")[-1]
UNSEEN = CFG["unseen"]  # evaluated only
# wiki40b, belebele, mC4, MMMLU (en = original MMLU), INCLUDE, MGSM, XNLI.  None = the benchmark lacks that language.
LCODE = {"en": ("en", "eng_Latn", "en", "EN", None, "en", "en"),
         "hi": ("hi", "hin_Deva", "hi", "HI_IN", "Hindi", None, "hi"),
         "fr": ("fr", "fra_Latn", "fr", "FR_FR", "French", "fr", "fr"),
         "es": ("es", "spa_Latn", "es", "ES_LA", "Spanish", "es", "es"),
         "de": ("de", "deu_Latn", "de", "DE_DE", "German", "de", "de"),
         "zh": ("zh-cn", "zho_Hans", "zh", "ZH_CN", "Chinese", "zh", "zh"),
         "ja": ("ja", "jpn_Jpan", "ja", "JA_JP", "Japanese", "ja", None),
         "ru": ("ru", "rus_Cyrl", "ru", None, "Russian", "ru", "ru")}
XNLI_WORDS = {"en": ("right", "Yes", "Also", "No"), "fr": ("correct", "Oui", "Aussi", "Non"),
              "es": ("correcto", "Sí", "Asi que", "No"), "de": ("richtig", "Ja", "Auch", "Nein"),
              "hi": ("सही", "हाँ", "इसलिए", "नहीं"), "zh": ("正确", "是的", "所以", "不是"),
              "ru": ("Правильно", "Да", "Так", "Нет")}
print("device:", DEV)


def wiki40b_text(t):
    if t[:2] in ("b'", 'b"'):  # Wiki-40B stores text as an escaped bytes literal
        t = ast.literal_eval(t).decode("utf-8")
    t = re.sub(r"_START_(ARTICLE|SECTION)_\n[^\n]*\n", "", t)
    return t.replace("_START_PARAGRAPH_\n", "").replace("_NEWLINE_", "\n").strip()


def token_chunks(texts, tok, seq, n):
    """Pack documents (EOS-separated, <= 2*seq tokens each) into n chunks of seq tokens."""
    buf, out = [], []
    for t in texts:
        buf += tok(t[:20000])["input_ids"][:2 * seq] + [tok.eos_token_id]
        while len(buf) >= seq and len(out) < n:
            out.append(buf[:seq])
            buf = buf[seq:]
        if len(out) == n:
            return torch.tensor(out)
    raise RuntimeError(f"only {len(out)}/{n} chunks available")


def optional(what, fn):
    try:
        return fn()
    except Exception as e:
        print(f"  WARNING: could not load {what} ({type(e).__name__}: {e}); continuing without it")
        return None


def load_bench(l):
    """Few-shot benchmarks: {name: {"shots": [...], "items": [...]}}. Shots never overlap the evaluated items."""
    from datasets import load_dataset
    w, b, m, mm, inc, mg, xn = LCODE[l]
    c, k, out = CFG, CFG["shots"], {}
    rnd = lambda: random.Random(0)  # fresh per benchmark: same Belebele order in every language
    mc = lambda q, opts, ans: dict(q=q, opts=[str(o) for o in opts], ans=int(ans))
    if mm == "EN":
        def f():
            dev = [mc(r["question"], r["choices"], r["answer"]) for r in load_dataset("cais/mmlu", "all", split="dev")]
            test = [mc(r["question"], r["choices"], r["answer"]) for r in load_dataset("cais/mmlu", "all", split="test")]
            rnd().shuffle(dev), rnd().shuffle(test)
            return dict(shots=dev[:k["mmmlu"]], items=test[:c["n_mmmlu"]])
        out["mmmlu"] = optional("MMLU (en)", f)
    elif mm:
        def f():
            rows = [mc(r["Question"], [r[x] for x in "ABCD"], "ABCD".index(str(r["Answer"]).strip()))
                    for r in load_dataset("openai/MMMLU", mm, split="test")]
            rnd().shuffle(rows)  # MMMLU has no dev split: the first rows become the shots and are not evaluated
            return dict(shots=rows[:k["mmmlu"]], items=rows[k["mmmlu"]:k["mmmlu"] + c["n_mmmlu"]])
        out["mmmlu"] = optional(f"MMMLU ({mm})", f)
    if inc:
        def f():
            row = lambda r: mc(r["question"], [r[f"option_{x}"] for x in "abcd"], r["answer"])
            test = [row(r) for r in load_dataset("CohereLabs/include-base-44", inc, split="test")]
            rnd().shuffle(test)
            shots = [row(r) for r in load_dataset("CohereLabs/include-base-44", inc, split="validation")][:k["include"]]
            return dict(shots=shots, items=test[:c["n_include"]])
        out["include"] = optional(f"INCLUDE ({inc})", f)
    if mg:
        out["mgsm"] = optional(f"MGSM ({mg})", lambda: dict(
            shots=[dict(r) for r in load_dataset("juletxara/mgsm", mg, split="train")][:k["mgsm"]],
            items=[dict(question=r["question"], answer=str(r["answer_number"])) for r in
                   load_dataset("juletxara/mgsm", mg, split="test")][:c["n_mgsm"]]))
    def f():
        rows = [dict(x) for x in load_dataset("facebook/belebele", b, split="test")]
        rnd().shuffle(rows)
        return dict(shots=rows[-k["belebele"]:], items=rows[:c["n_bele"]])
    out["belebele"] = optional(f"Belebele ({b})", f)
    if xn:
        def f():
            val = [dict(x) for x in load_dataset("facebook/xnli", xn, split="validation")]
            shots = [next(x for x in val if x["label"] == lab) for lab in (0, 2, 1)][:k["xnli"]]
            test = [dict(x) for x in load_dataset("facebook/xnli", xn, split="test")]
            rnd().shuffle(test)
            return dict(shots=shots, items=test[:c["n_xnli"]])
        out["xnli"] = optional(f"XNLI ({xn})", f)
    return {n: v for n, v in out.items() if v}


def load_data(tok):
    c = CFG
    if c["data_dir"].startswith("/content/drive"):
        from google.colab import drive
        drive.mount("/content/drive")
    key = json.dumps([SLUG, LANGS, UNSEEN] + [c[x] for x in ("seq", "n_train", "n_calib", "n_mc4", "n_wiki", "n_mmmlu",
                                                     "n_include", "n_mgsm", "n_bele", "n_xnli", "shots")], sort_keys=True)
    path = f"{c['data_dir']}/data_{SLUG}_{hashlib.md5(key.encode()).hexdigest()[:8]}.pt"
    if os.path.exists(path):
        print("using cached data", path)
        return torch.load(path, weights_only=False)
    print("Downloading data (first run only, ~5-15 min). Tip: add HF_TOKEN as a Colab/Kaggle secret to avoid rate limits.")
    from datasets import load_dataset
    d = {}
    for l in LANGS + UNSEEN:
        w, _, m, *_ = LCODE[l]
        t0 = time.time()
        train = token_chunks((wiki40b_text(x["text"]) for x in load_dataset("google/wiki40b", w, split="train", streaming=True)),
                             tok, c["seq"], c["n_train"] + c["n_calib"])
        d[l] = {"calib": train[:c["n_calib"]], "train": train[c["n_calib"]:],
                "mc4": token_chunks((x["text"] for x in load_dataset("allenai/c4", m, split="validation", streaming=True)),
                                    tok, c["seq"], c["n_mc4"]),
                "wiki": optional(f"Wiki-40B test ({w})", lambda: token_chunks(
                    (wiki40b_text(x["text"]) for x in load_dataset("google/wiki40b", w, split="test", streaming=True)),
                    tok, c["seq"], c["n_wiki"])),
                "bench": load_bench(l)}
        sizes = ", ".join(f"{n} {len(v['items'])}" for n, v in d[l]["bench"].items())
        print(f"  {l}: {sizes} ({time.time() - t0:.0f}s)", flush=True)
    os.makedirs(c["data_dir"], exist_ok=True)
    torch.save(d, path)
    return d


tok = AutoTokenizer.from_pretrained(CFG["model"])
tok.truncation_side = "left"  # over-long few-shot prompts lose their oldest shots, never the question
DATA = load_data(tok)

if CFG["math_frac"]:  # ablation: part of every language's TRAINING text (never calibration) becomes GSM8K-train text
    from datasets import load_dataset
    _n = int(CFG["math_frac"] * CFG["n_train"])
    _gsm = (f"Question: {r['question']}\nAnswer: {r['answer']}" for r in load_dataset("openai/gsm8k", "main", split="train"))
    _mt = token_chunks(_gsm, tok, CFG["seq"], _n * len(LANGS))  # MGSM = translated GSM8K TEST problems: no overlap
    for _i, _l in enumerate(LANGS):
        DATA[_l]["train"][-_n:] = _mt[_i * _n:(_i + 1) * _n]
    print(f"math-aware calibration: {_n} of {CFG['n_train']} training sequences per language are GSM8K-train text")


## Few-shot evaluation

In [ ]:
AMP_DTYPE = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16) if DEV == "cuda" else None


def amp():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if AMP_DTYPE else contextlib.nullcontext()


@torch.no_grad()
def token_logprobs(model, ids, mask=None, chunk=1024):
    with amp():
        h = model.model(input_ids=ids, attention_mask=mask).last_hidden_state[:, :-1]
    tgt = ids[:, 1:]
    lp = [-F.cross_entropy(model.lm_head(hc.float()).float(), tc, reduction="none")
          for hc, tc in zip(h.reshape(-1, h.shape[-1]).split(chunk), tgt.reshape(-1).split(chunk))]
    return torch.cat(lp).view_as(tgt)


@torch.no_grad()
def ppl(model, x, bs=4):
    nll, n = 0.0, 0
    for b in x.split(bs):
        b = b.to(DEV)
        nll -= token_logprobs(model, b).sum().item()
        n += b[:, 1:].numel()
    return math.exp(nll / n)


@torch.no_grad()
def letter_acc(model, prompts, answers, bs=8):
    """Pick A/B/C/D by the next-token logit after the prompt."""
    ids = [tok.encode(x)[0] for x in (" A", " B", " C", " D")]
    tok.padding_side, hits = "left", []
    for i in range(0, len(prompts), bs):
        enc = tok(prompts[i:i + bs], return_tensors="pt", padding=True, truncation=True, max_length=CFG["max_len"]).to(DEV)
        with amp():
            last = model.model(**enc).last_hidden_state[:, -1]
        pred = model.lm_head(last.float())[:, ids].argmax(-1).cpu()
        hits += (pred == torch.tensor(answers[i:i + bs])).int().tolist()
    return hits


def mc_text(x):
    return f"Question: {x['q'].strip()}\n" + "".join(f"{a}. {o.strip()}\n" for a, o in zip("ABCD", x["opts"])) + "Answer:"


def mc_eval(model, B, lang):  # MMMLU / MMLU / INCLUDE, k-shot
    head = "The following are multiple choice questions (with answers).\n\n"
    head += "".join(f"{mc_text(s)} {'ABCD'[s['ans']]}\n\n" for s in B["shots"])
    return letter_acc(model, [head + mc_text(x) for x in B["items"]], [x["ans"] for x in B["items"]])


def bele_text(x):
    return (f"P: {x['flores_passage']}\nQ: {x['question']}\nA: {x['mc_answer1']}\nB: {x['mc_answer2']}\n"
            f"C: {x['mc_answer3']}\nD: {x['mc_answer4']}\nAnswer:")


def bele_eval(model, B, lang):  # Belebele, k-shot
    head = "".join(f"{bele_text(s)} {'ABCD'[int(s['correct_answer_num']) - 1]}\n\n" for s in B["shots"])
    return letter_acc(model, [head + bele_text(x) for x in B["items"]], [int(x["correct_answer_num"]) - 1 for x in B["items"]])


@torch.no_grad()
def xnli_eval(model, B, lang, bs=4):  # XNLI, k-shot; picks the most likely of the three completed sentences
    q, *words = XNLI_WORDS[lang]
    head = "".join(f"{s['premise']}, {q}? {words[s['label']]}, {s['hypothesis']}\n" for s in B["shots"])
    tok.padding_side, hits = "right", []
    for i in range(0, len(B["items"]), bs):
        batch = B["items"][i:i + bs]
        enc = tok([f"{head}{x['premise']}, {q}? {w}, {x['hypothesis']}" for x in batch for w in words],
                  return_tensors="pt", padding=True, truncation=True, max_length=CFG["max_len"]).to(DEV)
        tl = token_logprobs(model, enc["input_ids"], enc["attention_mask"])
        pred = (tl * enc["attention_mask"][:, 1:]).sum(-1).cpu().view(len(batch), 3).argmax(-1)
        hits += (pred == torch.tensor([x["label"] for x in batch])).int().tolist()
    return hits


def prefix(s):
    """'Frage: ...' -> 'Frage:' (MGSM shots carry the question/answer words of their language)."""
    m = re.match(r"^[^:：]{1,40}[:：]", s)
    return m.group(0) if m else ""


@torch.no_grad()
def mgsm_eval(model, B, lang, bs=16):  # MGSM, k-shot chain of thought, greedy generation, exact final number
    qp, ap = prefix(B["shots"][0]["question"]), prefix(B["shots"][0]["answer"])
    head = "".join(f"{s['question'].strip()}\n{s['answer'].strip()}\n\n" for s in B["shots"])
    tok.padding_side, hits = "left", []
    for i in range(0, len(B["items"]), bs):
        batch = B["items"][i:i + bs]
        enc = tok([f"{head}{qp} {x['question'].strip()}\n{ap}" for x in batch], return_tensors="pt", padding=True,
                  truncation=True, max_length=CFG["max_len"]).to(DEV)
        with amp():
            gen = model.generate(**enc, max_new_tokens=CFG["mgsm_tokens"], do_sample=False, use_cache=True,
                                 stop_strings=["\n\n", qp] if qp else ["\n\n"], tokenizer=tok, pad_token_id=tok.pad_token_id)
        for x, g in zip(batch, tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)):
            nums = re.findall(r"-?\d[\d,]*(?:\.\d+)?", g.split("\n\n")[0])
            hits.append(int(bool(nums) and abs(float(nums[-1].replace(",", "")) - float(x["answer"].replace(",", ""))) < 1e-6))
    return hits


TASKS = {"mmmlu": mc_eval, "include": mc_eval, "mgsm": mgsm_eval, "belebele": bele_eval, "xnli": xnli_eval}


def evaluate(model):
    model.eval()
    r = {n: {} for n in ["mc4", "wiki", *TASKS, "hits"]}
    for l in LANGS + (UNSEEN if CFG["eval_unseen"] else []):
        t0, d = time.time(), DATA[l]
        r["mc4"][l] = round(ppl(model, d["mc4"]), 3)
        if d["wiki"] is not None:
            r["wiki"][l] = round(ppl(model, d["wiki"]), 3)
        for n, fn in TASKS.items():
            if n in d["bench"]:
                h = fn(model, d["bench"][n], l)
                r[n][l], r["hits"][f"{n}/{l}"] = round(sum(h) / len(h), 4), "".join(map(str, h))
        print(f"    eval {l}: {time.time() - t0:.0f}s", flush=True)
    r["copy_nll"] = round(copy_nll(model), 4)
    r["speed"] = speed(model)
    return r


@torch.no_grad()
def copy_nll(model, n=32, T=128):
    """Language-free in-context copying (induction) probe: random tokens shown twice; mean NLL on the repeat. The
    original model copies almost perfectly; a merge that breaks a cross-layer induction circuit does not."""
    x = torch.randint(1000, 100000, (n, T), generator=torch.Generator().manual_seed(1))
    return -token_logprobs(model, torch.cat([x, x], 1).to(DEV))[:, T:].mean().item()


@torch.no_grad()
def speed(model, T=1024, new=64):
    """Prefill tokens/s (batch 1, T tokens) and greedy decode tokens/s with a KV cache, on this device."""
    x = torch.randint(1000, 100000, (1, T), device=DEV)
    sync = torch.cuda.synchronize if DEV == "cuda" else (lambda: None)
    with amp():
        model.model(x[:, :64])
        sync()
        t0 = time.time()
        model.model(x)
        sync()
        pre = T / (time.time() - t0)
        t0 = time.time()
        model.generate(x[:, :64], max_new_tokens=new, min_new_tokens=new, do_sample=False, use_cache=True,
                       pad_token_id=tok.pad_token_id)
        sync()
    return dict(prefill_tok_s=round(pre), decode_tok_s=round(new / (time.time() - t0), 1))


## Original model, calibration states, planner

In [ ]:
teacher = AutoModelForCausalLM.from_pretrained(CFG["model"], dtype=torch.float32).to(DEV).eval()
teacher.config.use_cache = False
teacher.requires_grad_(False)
L = teacher.config.num_hidden_layers
if AMP_DTYPE:  # half-precision eval is ~4x faster on T4; switch it off if it changes the original model's perplexity
    x = DATA[LANGS[0]]["mc4"][:2]
    p16 = ppl(teacher, x)
    _a, AMP_DTYPE = AMP_DTYPE, None
    p32 = ppl(teacher, x)
    AMP_DTYPE = _a if abs(p16 / p32 - 1) < 0.02 else None
    print(f"mixed precision {AMP_DTYPE} (ppl fp32 {p32:.3f} vs half {p16:.3f})")


def flat(t):
    t = t[0] if isinstance(t, tuple) else t
    return t[:, 1:].reshape(-1, t.shape[-1]).float().cpu()  # position 0 (attention sink) excluded everywhere


@torch.no_grad()
def capture(model, li, inputs=(), outputs=()):
    """Inputs / outputs of submodules of layer li ("" = the layer itself) on the current calibration tokens of all
    languages. Output keys get a ':out' suffix."""
    layer, got, hooks = model.model.layers[li], {}, []
    for n in inputs:
        hooks.append(layer.get_submodule(n).register_forward_pre_hook(lambda m, a, n=n: got.setdefault(n, []).append(flat(a[0]))))
    for n in outputs:
        hooks.append(layer.get_submodule(n).register_forward_hook(lambda m, a, o, n=n: got.setdefault(n + ":out", []).append(flat(o))))
    try:
        for l in LANGS:
            with amp():
                model.model(DATA[l]["calib"][:NCAL].to(DEV))
    finally:
        for h in hooks:
            h.remove()
    return {k: torch.cat(v) for k, v in got.items()}


@torch.no_grad()
def layer_inputs():
    got, hooks = {i: [] for i in range(L)}, []
    for i in range(L):
        hooks.append(teacher.model.layers[i].register_forward_pre_hook(lambda m, a, i=i: got[i].append(flat(a[0]))))
    for l in LANGS:
        teacher.model(DATA[l]["calib"].to(DEV))  # fp32: these are the fitting targets
    for h in hooks:
        h.remove()
    return [torch.cat(got[i]) for i in range(L)]


n_cal, S = DATA[LANGS[0]]["calib"].shape
HS = layer_inputs()  # HS[i] = original hidden state entering layer i, [calibration tokens, d]
SEQ_ID = torch.arange(n_cal).repeat_interleave(S - 1).repeat(len(LANGS))
TOK_LANG = torch.arange(len(LANGS)).repeat_interleave(n_cal * (S - 1))


def set_calib(n):
    """Use the first n calibration sequences per language; the n-th is held out to check every fit."""
    global NCAL, ROWS, HOLD, FIT
    NCAL, ROWS = n, SEQ_ID < n
    HOLD = SEQ_ID[ROWS] == n - 1
    FIT = ~HOLD


set_calib(n_cal)
hs = lambda i: HS[i][ROWS]


def ridge(X, Y, lam=None):
    """argmin_W ||X W - Y||^2 + lam*mean(diag(X^T X))*||W||^2 in float64."""
    dev = DEV if DEV == "cuda" else "cpu"  # MPS has no float64
    X, Y = X.to(dev, torch.float64), Y.to(dev, torch.float64)
    G = X.T @ X
    lam = CFG["ridge"] if lam is None else lam
    return torch.linalg.solve(G + lam * G.trace() / len(G) * torch.eye(len(G), dtype=G.dtype, device=dev), X.T @ Y).float().cpu()


def rel_err(P, T):
    """Held-out relative mismatch to the original model's hidden state."""
    return ((P[HOLD] - T[HOLD]).norm() / T[HOLD].norm()).item()


def angular(a, b):
    return (torch.arccos(F.cosine_similarity(a, b, -1).clamp(-1, 1)).mean() / math.pi).item()


def plan_pairs(k):
    """k non-overlapping adjacent pairs whose input and output states are most similar (Gromov et al. angular distance)."""
    p = CFG["protect"]
    score = {i: angular(HS[i], HS[i + 2]) for i in range(p, L - p - 1)}
    chosen = []
    for i in sorted(score, key=score.get):
        if all(abs(i - j) > 1 for j in chosen):
            chosen.append(i)
        if len(chosen) == k:
            break
    return [[i, i + 1] for i in sorted(chosen)]


def with_layers(layers):
    """Shallow copy of the original model that shares everything except the layer list."""
    s = copy.copy(teacher)
    s._modules = dict(teacher._modules)
    m = copy.copy(teacher.model)
    m._modules = dict(teacher.model._modules)
    m.layers = nn.ModuleList(layers)
    s.model = m
    return s


def assemble(plan, merged):
    """Each group of original layers in plan is replaced by one layer (merged[gi])."""
    layers, i, gi = [], 0, 0
    while i < L:
        if gi < len(plan) and i == plan[gi][0]:
            layers.append(merged[gi])
            i, gi = plan[gi][-1] + 1, gi + 1
        else:
            layers.append(teacher.model.layers[i])
            i += 1
    return with_layers(layers)


def student_index(plan, gi):
    return plan[gi][0] - sum(len(g) - 1 for g in plan[:gi])


print(f"{SLUG}: {L} layers, {len(HS[0])} calibration tokens ({int(HOLD.sum())} held out)")

In [ ]:
REF_PATH = f"{CFG['data_dir']}/results_reference_{SLUG}.json"  # original + drop rows, shared by all notebooks
os.makedirs(CFG["data_dir"], exist_ok=True)
load_json = lambda p: json.load(open(p)) if os.path.exists(p) else {}
pstr = lambda plan: "; ".join(f"{g[0]}-{g[-1]}" for g in plan) or "—"
done = lambda path, method, k, plan: f"{method}|{k}|{pstr(plan)}" in load_json(path)


def record(path, method, k, plan, model, build_s=0.0):
    t0 = time.time()
    e = evaluate(model)
    R = load_json(path)
    R[f"{method}|{k}|{pstr(plan)}"] = dict(method=method, k=k, plan=pstr(plan), layers=len(model.model.layers),
                                           params=sum(p.numel() for p in model.parameters()), build_s=round(build_s, 1), eval=e)
    json.dump(R, open(path, "w"), indent=1)
    print(f"{method} k={k} [{pstr(plan)}]: eval {time.time() - t0:.0f}s", flush=True)


def run_all(algo, build, plans=None):
    """Original + `drop` reference (keep the first layer of each group) + the algorithm, for every merge count."""
    algo = algo + CFG["tag"]
    res = f"{CFG['data_dir']}/results_{algo}_{SLUG}.json"
    if not done(REF_PATH, "original", 0, []):
        record(REF_PATH, "original", 0, [], teacher)
    for k in CFG["merge_counts"]:
        plan = plans[k] if plans else plan_pairs(k)
        print(f"\n=== {k} layers removed: groups {pstr(plan)} ===", flush=True)
        if not done(REF_PATH, "drop", k, plan):
            record(REF_PATH, "drop", k, plan, assemble(plan, [teacher.model.layers[g[0]] for g in plan]))
        if not done(res, algo, k, plan):
            t0 = time.time()
            m = build(plan)
            record(res, algo, k, plan, m, time.time() - t0)
            del m
            gc.collect()
            if DEV == "cuda":
                torch.cuda.empty_cache()
    return res


def table(paths=None):
    """One row per (method, layers removed, plan). PPL: mean % increase over languages (and the worst language).
    Accuracy: mean over languages, in %. `avg acc` = mean of the five few-shot benchmarks."""
    paths = paths or sorted(glob.glob(f"{CFG['data_dir']}/results_*_{SLUG}.json"))
    R = {}
    for p in paths:
        R.update(load_json(p))
    o = R["original|0|—"]
    rows = []
    for r in R.values():
        e, row = r["eval"], dict(method=r["method"], removed=r["k"], plan=r["plan"],
                                 **{"param red %": round(100 * (1 - r["params"] / o["params"]), 1)})
        for ds in ("mc4", "wiki"):
            if e[ds]:
                inc = [100 * (e[ds][l] / o["eval"][ds][l] - 1) for l in e[ds] if l in LANGS]
                row[f"{ds} ppl +%"] = round(sum(inc) / len(inc), 1)
                row[f"{ds} worst +%"] = round(max(inc), 1)
        accs = []
        for ds in TASKS:
            if e.get(ds):
                v = [e[ds][l] for l in LANGS if l in e[ds]]
                row[ds] = round(100 * sum(v) / len(v), 1)
                accs.append(row[ds])
        row["avg acc"] = round(sum(accs) / max(1, len(accs)), 1)
        un = [e[ds][l] for ds in TASKS if e.get(ds) for l in UNSEEN if l in e[ds]]
        if un:
            row["unseen acc"] = round(100 * sum(un) / len(un), 1)  # ja / ru, never trained on
        if "copy_nll" in e:
            row["copy nll"] = e["copy_nll"]
            row["decode tok/s"] = e["speed"]["decode_tok_s"]
        row["build s"] = r["build_s"]
        rows.append(row)
    df = pd.DataFrame(rows).sort_values(["removed", "avg acc"], ascending=[True, False]).reset_index(drop=True)
    pd.set_option("display.width", 250)
    pd.set_option("display.max_columns", 30)
    return df

## Shared distillation code

In [ ]:
ALPHA = {"v": 1.0}  # 1 = original group, 0 = merged layer only (used by Fade / Blend)
TRAIN_DTYPE = AMP_DTYPE


def outputs_of(model, idx):
    got = {}
    return got, [model.model.layers[i].register_forward_hook(lambda m, a, o, i=i: got.__setitem__(i, o)) for i in idx]


def kd_losses(student, x, t_out, s_out, pairs):
    ctx = lambda: torch.autocast("cuda", dtype=TRAIN_DTYPE) if TRAIN_DTYPE else contextlib.nullcontext()
    with torch.no_grad(), ctx():
        th = teacher.model(x).last_hidden_state
    with ctx():
        sh = student.model(x).last_hidden_state
    hid = sum((s_out[si][:, 1:].float() - t_out[ti][:, 1:].float()).pow(2).mean() / t_out[ti][:, 1:].float().pow(2).mean()
              for si, ti in pairs) / len(pairs)
    kl, d = 0.0, th.shape[-1]
    for a, b in zip(sh.reshape(-1, d).split(512), th.reshape(-1, d).split(512)):  # chunks: full-vocab logits are big
        with ctx():
            ls, lt = teacher.lm_head(a), teacher.lm_head(b)
        kl = kl + F.kl_div(F.log_softmax(ls.float(), -1), F.log_softmax(lt.float(), -1), log_target=True, reduction="sum")
    return kl / x.numel(), hid


def kd_train(student, pairs, params):
    """Language-balanced distillation (one sequence per language per step): KL(original || student) on next-token
    distributions + normalized hidden-state MSE at each (student layer, original layer) pair. ALPHA follows a cosine
    from 1 to 0 over the first `anneal_frac` of the steps, then stays 0."""
    t_out, th = outputs_of(teacher, [t for _, t in pairs])
    s_out, sh = outputs_of(student, [s for s, _ in pairs])
    opt = torch.optim.AdamW(params, lr=CFG["lr"], weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=TRAIN_DTYPE == torch.float16)
    N, A = CFG["steps"], int(CFG["anneal_frac"] * CFG["steps"])
    gen = torch.Generator().manual_seed(CFG["seed"])
    batch = lambda: torch.stack([DATA[l]["train"][torch.randint(len(DATA[l]["train"]), (1,), generator=gen)[0]]
                                 for l in LANGS]).to(DEV)
    try:
        ALPHA["v"] = 1.0
        with torch.no_grad():
            kl, hid = kd_losses(student, batch(), t_out, s_out, pairs)
        assert kl.item() < 1e-3 and hid.item() < 1e-3, f"at alpha=1 the student must equal the original (kl {kl}, hid {hid})"
        t0 = time.time()
        for step in range(N):
            ALPHA["v"] = 0.5 * (1 + math.cos(math.pi * (step + 1) / A)) if step + 1 < A else 0.0  # < 1 from step 0: the merged layer gets gradient
            kl, hid = kd_losses(student, batch(), t_out, s_out, pairs)
            loss = kl + CFG["hid_w"] * hid
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(opt)
            scaler.update()
            if step % 50 == 0 or step == N - 1:
                print(f"  step {step:4d}  alpha {ALPHA['v']:.3f}  kl {kl.item():.4f}  hid {hid.item():.4f}  ({time.time() - t0:.0f}s)",
                      flush=True)
    finally:
        for h in th + sh:
            h.remove()
        ALPHA["v"] = 0.0

## Algorithm: AAM

In [ ]:
ALGO = f"aam_anneal{CFG['anneal_frac']}"


class Fade(nn.Module):
    """A layer being absorbed: its residual update is scaled by ALPHA (1 = original, 0 = removed)."""

    def __init__(self, inner):
        super().__init__()
        self.inner = inner

    def forward(self, h, *a, **k):
        return h if ALPHA["v"] == 0 else h + ALPHA["v"] * (self.inner(h, *a, **k) - h)


def pick_anchor(g):
    T = hs(g[-1] + 1)
    errs = {a: rel_err(capture(assemble([g], [teacher.model.layers[a]]), g[0], [], [""])[":out"], T) for a in g}
    return min(errs, key=errs.get), errs


def build(plan):
    layers, anchors = list(teacher.model.layers), []
    for g in plan:
        a, errs = pick_anchor(g)
        print(f"  group {pstr([g])}: held-out mismatch when kept alone " + ", ".join(f"L{j} {e:.1%}" for j, e in errs.items())
              + f" -> keep L{a}")
        for j in g:
            layers[j] = copy.deepcopy(teacher.model.layers[j]).requires_grad_(True) if j == a else Fade(teacher.model.layers[j])
        anchors.append(layers[a])
    idx = [g[-1] for g in plan]  # during training the student still has every layer: indices match the original
    kd_train(with_layers(layers), list(zip(idx, idx)), [p for m in anchors for p in m.parameters()])
    for m in anchors:
        m.requires_grad_(False)
    return assemble(plan, anchors)  # faded layers are gone: one layer per group

## Run and results
Finished runs are saved and skipped on re-run.

In [ ]:
RES = run_all(ALGO, build, globals().get("PLANS"))  # PLANS: searched groups (notebook 04); else angular-distance pairs
print("This algorithm vs the references:")
display(table([REF_PATH, RES]))
print("Every algorithm whose results are in", CFG["data_dir"], "(run the other notebooks with the same data_dir):")
display(table())